# Fine-Tune a Model for Real (Free Colab)

Run a small financial-sentiment fine-tune end-to-end on a free T4 GPU and get **real** metrics.

**First:** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom.

Each code cell has a short note above it explaining what it does and why.

### 1. Confirm you have a GPU
Training needs a GPU. If this errors or shows nothing, fix the runtime type first (see above).

In [ ]:
!nvidia-smi

### 2. Get the code (fresh clone every time)
Deletes any stale copy first, then clones the current version from GitHub so the latest scripts are always present. The `ls` prints what's in `scripts/` so you can confirm `predict.py` is there.

In [ ]:
%cd /content
!rm -rf LLM-Fine-Tuning-with-QLoRA-and-DPO
!git clone https://github.com/SoumithReddy6/LLM-Fine-Tuning-with-QLoRA-and-DPO.git
%cd LLM-Fine-Tuning-with-QLoRA-and-DPO
!ls scripts/

### 3. Install a known-good, compatible stack
Floating versions break. These pins are mutually compatible and match the training scripts' API. **`torch` and `torchvision` must match** — that pairing is the most common setup failure.

*(Takes 2-3 minutes. Ignore pip dependency warnings. After this finishes, do `Runtime → Restart session`, then Run all from the top.)*

In [ ]:
!pip -q install \
  "torch==2.4.1" \
  "torchvision==0.19.1" \
  "transformers==4.46.3" \
  "trl==0.11.4" \
  "peft==0.13.2" \
  "accelerate==1.1.1" \
  "datasets==3.1.0" \
  "bitsandbytes==0.44.1"

### 4. Build the real dataset
Turns the Financial PhraseBank benchmark into train / test / preference files. The test set is held out, so your accuracy is measured on data the model never trained on — that's what makes the number real.

In [ ]:
!python3 scripts/prepare_financial_phrasebank.py --max-train 2000 --test-size 0.2 --seed 7

### 5. Measure the BASELINE first (base model, no fine-tuning)
To prove fine-tuning helped, you need a "before" number. This runs the raw base model zero-shot on the test set. Expect modest accuracy and some messy / invalid outputs — that's the point.

In [ ]:
!python3 scripts/predict.py \
  --base-model Qwen/Qwen2.5-0.5B-Instruct \
  --input data/financial_test.jsonl \
  --output artifacts/baseline_preds.jsonl \
  --load-in-4bit

### 6. Supervised fine-tune (QLoRA / SFT)
This is the actual training. It teaches the model to output clean financial-sentiment labels. On a T4 with 2,000 examples this takes a few minutes. You'll see the loss printed as it learns — it should trend down.

In [ ]:
!python3 scripts/train_sft.py \
  --config configs/experiments/qwen25_0_5b_qlora_rank16.yaml \
  --local-data data/financial_train.jsonl

### 7. Get the fine-tuned model's predictions
Same script as the baseline, but now with `--adapter` pointing at what you just trained. This is the "after" number.

In [ ]:
!python3 scripts/predict.py \
  --base-model Qwen/Qwen2.5-0.5B-Instruct \
  --adapter outputs/qwen25_0_5b_qlora_rank16 \
  --input data/financial_test.jsonl \
  --output artifacts/tuned_preds.jsonl \
  --load-in-4bit

### 8. Score it: real accuracy, F1, calibration, and the lift over baseline
Computes the honest metrics and the improvement fine-tuning bought. The `relative_accuracy_lift` field is your headline result.

In [ ]:
!python3 scripts/evaluate_model.py --config configs/experiments/eval_qwen25_0_5b.yaml

---
## Optional: DPO (preference tuning)

DPO further nudges the model toward preferred answers. Run these after the steps above.

### 9. Train DPO from the SFT adapter

In [ ]:
!python3 scripts/train_dpo.py --config configs/experiments/qwen25_0_5b_dpo.yaml

### 10. Predict + score the DPO model

In [ ]:
!python3 scripts/predict.py \
  --base-model Qwen/Qwen2.5-0.5B-Instruct \
  --adapter outputs/qwen25_0_5b_dpo \
  --input data/financial_test.jsonl \
  --output artifacts/dpo_preds.jsonl \
  --load-in-4bit

import json, sys
sys.path.insert(0, "src")
from qlora_dpo_finance.data import read_jsonl
from qlora_dpo_finance.metrics import evaluate_predictions
metrics = evaluate_predictions(read_jsonl("artifacts/dpo_preds.jsonl"), read_jsonl("artifacts/baseline_preds.jsonl"))
print(json.dumps(metrics, indent=2))

### 11. Save your real results
Download the metrics so you can commit them. These replace the fake hand-authored smoke numbers with results a model actually produced.

In [ ]:
from google.colab import files
files.download("artifacts/eval_qwen25_0_5b.json")
files.download("artifacts/tuned_preds.jsonl")

---
## If a cell errors

- **`operator torchvision::nms does not exist`** — torch/torchvision version mismatch. Make sure step 3 includes the matching `torchvision==0.19.1`, then `Runtime → Restart session` and Run all.
- **"CUDA out of memory"** — lower `per_device_train_batch_size` in the config (try 4, then 2), or use `--max-train 1000` in step 4.
- **A `TypeError` from `SFTTrainer` / `DPOTrainer`** — a TRL version mismatch. Re-run step 3 exactly (the pins matter), then `Runtime → Restart session` and run from the top.
- **Anything else** — copy the full error text and send it over. Debugging a real training run is normal, and it's exactly the experience interviewers ask about.

## Scaling up later
The 7B/8B configs use the **identical method** — only the model size and a larger GPU differ. Once the small run works, swapping the model name is the only change. For Llama 3, request model access on Hugging Face and run `huggingface-cli login` first.